# Homework assignment 1: Markov Chain

## Problem 1
Read the article in https://www.geeksforgeeks.org/markov-chains-in-nlp/, and answer the following questions.

* (a) What are N-grams of an input sequence?
    * [A contiguous sequence of n items (words or characters) from a given sample of text.]

* (b) How to determine the probability of each element in the transition matrix?
    * [We calculate these probabilities by counting the number of times a particular word appears after another word in the N-grams.]

* (c) If you want to increase the variety of the sequence generation (every time the outputs are different), what kinds of properties the training texts should be?
    *   1. Diversity
        2. Complexity
        3. Length Variation
        4. Contextual Richness
        5. Repetition and Redundancy
        6. Multilingual Context
        7. Thematic Variation
        8. Interactive Dialogue

## Problem 2
Try the following codes, and answer questions.

In [ ]:
# Install the required packages
!pip install nltk
!pip install numpy

In [ ]:
import re
from nltk import ngrams
import numpy as np
from typing import List, Tuple, Union, Dict
from itertools import product
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
def words_to_index(words: List[str], base: int, unique_words: Dict[str, int]) -> int:
    """
        Convert a list of words to an index representation.

        Args:
            words (List[str]): The list of words to be converted.
            base (int): The base value used for conversion.
            unique_words (Dict[str, int]): A dictionary mapping unique words to their corresponding indices.

        Returns:
            - int: The index representation of the given list of words.

        Example:
            >>> words_to_index(["the", "quick", "brown", "fox"], 10, {"the": 0, "quick": 1, "brown": 2, "fox": 3})
            123
    """
    length = len(words)
    numbers = [unique_words[word] for word in words]
    return sum([num * (base ** (length - 1 - idx)) for idx, num in enumerate(numbers)])

def index_to_words(index: int, base: int, length: int, unique_words: Dict[str, int]):
    """
        Converts an index to a list of words based on a given base and unique words.

        Args:
            index (int): The index to convert.
            base (int): The base used for conversion.
            length (int): The length of the resulting list.
            unique_words (Dict[str, int]): A dictionary mapping unique words to their corresponding indices.

        Returns:
            List[str]: A list of words corresponding to the given index.

        Examples:
            >>> unique_words = {'apple': 0, 'banana': 1, 'cherry': 2}
            >>> index_to_words(5, 3, 2, unique_words)
            ['banana', 'cherry'] (Because 5 = 1 * 3^1 + 2 * 2^0)

            >>> unique_words = {'red': 0, 'green': 1, 'blue': 2}
            >>> index_to_words(2, 3, 1, unique_words)
            ['blue'] (Because 2 = 2 * 3^0)
    """
    numbers = []

    unique_words_list = list(unique_words.keys())
    for pow in range(length - 1, -1, -1):
        numbers.append(index // (base ** pow))
        index -= numbers[-1] * (base ** pow)

    return [unique_words_list[num] for num in numbers]

### Step 1: Remove some unnecessary characters

In [ ]:
def remove_unnecessary_characters(text: str) -> str:
    """
        Removes unnecessary characters from the given text and converts it to lowercase.

        Args:
            text (str): The input text to be processed.
        Returns:
            str: The processed text with unnecessary characters removed and converted to lowercase.
    """
    # Remove unnecessary characters
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

    # Convert to lowercase
    text = text.lower()

    print(f"Processed Text: {text}")


    return text

### Step 2: Generate N-grams

In [ ]:
import nltk
nltk.download('punkt')
from nltk.util import ngrams
from typing import List, Tuple

def generate_n_grams(text: str, n: int) -> List[Tuple[str]]:
    """
    Generate n-grams from the given text.

    Args:
        text (str): The input text from which n-grams will be generated.
        n (int): The number of consecutive words in each n-gram.

    Returns:
        List[Tuple[str]]: A list of tuples representing the generated n-grams.
    """
    # Tokenize the input text into words
    words = nltk.word_tokenize(text)

    # Generate n-grams using the nltk ngrams function
    n_grams = ngrams(words, n)

    # Convert to list of tuples
    n_grams_list = list(n_grams)

    return n_grams_list


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Step 3: Compute Transition Matrix

In [ ]:
def compute_transition_matrix(n_grams: List[Tuple[str]]):
    """
    Computes the transition matrix and unique states for a given list of n-grams.

    Args:
        n_grams (List[Tuple[str]]): A list of n-grams, where each n-gram is a tuple of strings.

    Returns:
        np.ndarray: transition matrix.
        Dict[str, int]: A dictionary mapping unique words to their corresponding indices.
    """

    # Get the value of n
    n = len(n_grams[0])

    ## Step 3.1: Collect all possible words (label words with indices)

    unique_words = {}
    unique_words_count = 0

    # A helper function to add a word to the unique_words dictionary, if it is not already present
    def _add_to_unique_word(word: str):
        nonlocal unique_words_count
        if unique_words.get(word, None) is None:
            unique_words[word] = unique_words_count
            unique_words_count += 1

    # Iterate over all n-grams to collect all possible words
    for n_gram in n_grams:
        for word in n_gram:
            _add_to_unique_word(word)  # Add the word to the unique_words dictionary

    ## Step 3.2: Compute the transition matrix

    def words_to_index(words: Tuple[str], unique_words_count: int, unique_words: Dict[str, int]):
        """Converts a tuple of words into an index based on unique_words."""
        index = 0
        for i, word in enumerate(words):
            index += unique_words[word] * (unique_words_count ** (len(words) - i - 1))
        return index

    def index_to_words(index: int, unique_words_count: int, word_length: int, unique_words: Dict[str, int]):
        """Converts an index into a tuple of words based on unique_words."""
        words = []
        reverse_dict = {v: k for k, v in unique_words.items()}
        for i in range(word_length):
            quotient = index // (unique_words_count ** (word_length - i - 1))
            words.append(reverse_dict[quotient])
            index = index % (unique_words_count ** (word_length - i - 1))
        return tuple(words)

    # Compute the number of states in the Markov chain
    unique_states_count = unique_words_count ** (n - 1)

    # Create transition matrix, filled with zeros
    transition_matrix = np.zeros((unique_states_count, unique_states_count))

    # Count the number of transitions from each state to another state
    for n_gram in n_grams:
        state_from = n_gram[:n-1]  # first n-1 words
        state_to = n_gram[1:]  # last n-1 words

        state_from_index = words_to_index(state_from, unique_words_count, unique_words)
        state_to_index = words_to_index(state_to, unique_words_count, unique_words)

        # Increment the count of the transition from state_from to state_to
        transition_matrix[state_from_index][state_to_index] += 1

    # Handle special case where sum of transitions of a state is zero
    row_sums_is_zero = transition_matrix.sum(axis=1) == 0
    first_word = list(unique_words.keys())[0]

    for idx, is_zero in enumerate(row_sums_is_zero):
        if is_zero:
            state_name = index_to_words(idx, unique_words_count, n-1, unique_words)
            possible_states_start_index = words_to_index(state_name[1:] + (first_word,), unique_words_count, unique_words)
            possible_states_end_index = possible_states_start_index + unique_words_count
            transition_matrix[idx, possible_states_start_index:possible_states_end_index] = 1

    ## Step 3.3: Normalize the transition matrix

    # Compute the sum of each row
    row_sums = transition_matrix.sum(axis=1, keepdims=True)

    # Avoid division by zero for rows with sum 0 by keeping them as 0 (they have no transitions)
    row_sums[row_sums == 0] = 1

    # Divide each element by the sum of its row
    transition_matrix /= row_sums

    return transition_matrix, unique_words


### Step 4: Generate text

In [ ]:
def generate(unique_words: Dict[str, int], transition_matrix: np.ndarray, start_from: Union[str, List[str]], n: int, length: int=10):
    """
        Generate text using a Markov chain model.

        Args:
            unique_states (Dict[str, int]): A dictionary mapping unique words to their corresponding indices.
            transition_matrix (np.ndarray): A numpy array representing the transition probabilities between states.
            start_from (Union[str, List[str]]): The starting state(s) for text generation.
            n (int): The size of the grams.
            length (int, optional): The length of the generated text. Defaults to 10.

        Returns:
            The generated text.
    """
    # Generate text
    generated_words = start_from.copy() if type(start_from) is list else [start_from]
    generated_words = [word.lower() for word in generated_words]  # Convert to lowercase

    # Add a check to ensure all words in start_from exist in unique_words
    for word in generated_words:
        if word not in unique_words:
            raise KeyError(f"Word '{word}' is not found in unique_words.")


    # Assert if the number of start words does not equal to n-1
    assert len(generated_words) >= n-1, "The number of start words should be greater than or equals to n-1 ({})".format(n-1)

    # Get the number of unique words
    unique_words_count = len(unique_words)

    # [TODO] Get the number of unique states
    # hint: check step 3.2 in the compute_transition_matrix function
    unique_states_count = unique_words_count ** (n - 1)

    # Generate the next words
    for _ in range(length):
        # [TODO] Get index of current states
        # hint: The current states (current words) is the last n-1 words in the generated text
        # hint: use words_to_index function
        current_words_index = words_to_index(generated_words[-(n - 1):], unique_words_count, unique_words)

        # [TODO] Get probability distribution for next state, using the information in the transition matrix
        probabilities = transition_matrix[current_words_index]

        # Select next word based on probabilities, using np.random.choice function
        next_words_index = np.random.choice(unique_states_count, p=probabilities)

        # [TODO] Decode the index and get the last word
        # hint: use index_to_words function
        next_word = index_to_words(next_words_index, unique_words_count, n - 1, unique_words)[-1]


        # Add next word to generated text
        generated_words.append(next_word)

    # return generated string
    return ' '.join(generated_words)

In [ ]:
# [TODO] Change the text below and try different values of n (b)
text = """Amidst the vibrant city, performers dazzled crowds with unique talents, from juggling torches to enchanting melodies."""
n = 3

# Process the text and generate the transition matrix
text = remove_unnecessary_characters(text)
n_grams = generate_n_grams(text, n)
transition_matrix, unique_words = compute_transition_matrix(n_grams)

Processed Text: amidst the vibrant city performers dazzled crowds with unique talents from juggling torches to enchanting melodies


In [ ]:
# Print the transition matrix and unique states for obsevation
print("===== The indices for unique states are: =====")
unique_words_count = len(unique_words)
for word_name in list(product(*[unique_words for _ in range(n-1)]))[:20]:
    print(f"{','.join(word_name):10s}: {words_to_index(word_name, unique_words_count, unique_words)}")
print("...", end="\n\n")

print("===== The transition matrix is (Shape of trasition matrix: {}): =====".format(transition_matrix.shape))
print(transition_matrix)
print()

===== The indices for unique states are: =====
amidst,amidst: 0
amidst,the: 1
amidst,vibrant: 2
amidst,city: 3
amidst,performers: 4
amidst,dazzled: 5
amidst,crowds: 6
amidst,with: 7
amidst,unique: 8
amidst,talents: 9
amidst,from: 10
amidst,juggling: 11
amidst,torches: 12
amidst,to : 13
amidst,enchanting: 14
amidst,melodies: 15
the,amidst: 16
the,the   : 17
the,vibrant: 18
the,city  : 19
...

===== The transition matrix is (Shape of trasition matrix: (256, 256)): =====
[[0.0625 0.0625 0.0625 ... 0.     0.     0.    ]
 [0.     0.     0.     ... 0.     0.     0.    ]
 [0.     0.     0.     ... 0.     0.     0.    ]
 ...
 [0.     0.     0.     ... 0.     0.     0.    ]
 [0.     0.     0.     ... 0.     0.     0.    ]
 [0.     0.     0.     ... 0.0625 0.0625 0.0625]]



In [ ]:
# [TODO] Write down 3 or more initial words and length of generated text to start the text generation
experiments = [
    ('Amidst the vibrant', 10),
    ('performers dazzled crowds', 15),
    ('juggling torches to', 30)
]

for idx, (start_from, length) in enumerate(experiments, 1):
    start_from = start_from.split(" ")

    # Generate text using the transition matrix
    generated_text = generate(unique_words, transition_matrix, start_from, n, length=length)

    # Print out the generated text
    print("#{} (length={}): {}".format(idx, length, generated_text))

#1 (length=10): amidst the vibrant city performers dazzled crowds with unique talents from juggling torches
#2 (length=15): performers dazzled crowds with unique talents from juggling torches to enchanting melodies from the juggling torches to enchanting
#3 (length=30): juggling torches to enchanting melodies dazzled crowds with unique talents from juggling torches to enchanting melodies enchanting torches juggling performers amidst performers melodies melodies talents the enchanting juggling to melodies crowds dazzled to


### Answer the following questions

* (a) Write a new text of at least 15 words as the input.
    * Amidst the vibrant city, performers dazzled crowds with unique talents, from juggling torches to enchanting melodies.

* (b) Run the program 3 times with different output length and different initial words. Show the outputs.
    * (Please modify the `experiments` at the cell above and run the cell)
    
    experiments = [

    ('Amidst the vibrant', 10),

    ('performers dazzled crowds', 15),

    ('juggling torches to', 30)
]


    #1 (length=10): amidst the vibrant city performers dazzled crowds with unique talents from juggling torches

    #2 (length=15): performers dazzled crowds with unique talents from juggling torches to enchanting melodies from the juggling torches to enchanting
    
    #3 (length=30): juggling torches to enchanting melodies dazzled crowds with unique talents from juggling torches to enchanting melodies enchanting torches juggling performers amidst performers melodies melodies talents the enchanting juggling to melodies crowds dazzled to


* (c) Try different N of N-grams. How the N influences the output sequence?
    *  
    As N increases, the model uses longer contexts to predict the next word, leading to more diverse and less repetitive sequences. However, larger n-grams can also introduce more randomness and less fluent transitions, as the context window becomes larger and more specific.

## Problem 3
The Stationary Distribution of a Markov chain is a distribution of probabilities that remains unchanged after a transition from one state to another.

* (a) Ask an LLM (Large Language Model), such as ChatGPT, what are the applications of stationary distribution of a Markov chain. You need to show which prompts are used, and state how you verify the correctness of the results (output by LLMs).
   
   

### Applications of Stationary Distribution

1. **Queueing Theory**:
   - Used to analyze systems such as network routers and call centers where customers arrive and are served.
   - The stationary distribution helps predict long-term average wait times and system utilization.

2. **Economics**:
   - Applied in economic modeling to study the distribution of wealth or resources in a population.
   - Helps in understanding how economies reach equilibrium states.

3. **Genetics**:
   - Used to model allele frequencies in populations over generations (Wright-Fisher model).
   - The stationary distribution indicates the long-term behavior of genetic traits.

4. **PageRank Algorithm**:
   - Google’s PageRank algorithm utilizes stationary distributions of Markov chains to rank web pages.
   - The stationary distribution represents the probability of a user landing on a particular page after many random clicks.

5. **Social Networks**:
   - In modeling social interactions and information diffusion, the stationary distribution can show how information spreads through a network over time.

6. **Statistical Physics**:
   - Used to describe the equilibrium state of physical systems, like gases, where particles move in a random manner.

7. **Machine Learning**:
   - In reinforcement learning, particularly in policy evaluation and understanding the behavior of agents in environments.

### Verification of Results

To verify the correctness of results involving stationary distributions, the following methods can be employed:

1. **Mathematical Proofs**:
   - Show that the distribution satisfies the balance equations: ∑pi P = pi), where pi is the stationary distribution and P is the transition matrix.
   - Prove that ∑i pi = 1.

2. **Simulations**:
   - Run simulations of the Markov chain over a large number of steps and observe the distribution of states. Compare this empirical distribution with the theoretical stationary distribution.

3. **Convergence Checks**:
   - Analyze how quickly the distribution approaches the stationary distribution over time by checking convergence metrics (e.g., total variation distance).

4. **Numerical Methods**:
   - Use numerical algorithms to compute the stationary distribution directly (e.g., power iteration method or solving linear equations).

5. **Sensitivity Analysis**:
   - Alter parameters in the Markov chain (like transition probabilities) and observe how the stationary distribution changes to ensure robustness.



### Prompt

- "What is the stationary distribution of a Markov chain representing a queueing system with exponential service times?"
- "How does the PageRank algorithm utilize the stationary distribution to rank web pages?"

 How I verified:

  Started by searching the methods mentioned online including and typing in " professional papers".  One of them I found is:
  https://math.uchicago.edu/~may/VIGRE/VIGRE2007/REUPapers/FINALFULL/Volfovsky.pdf

=========================================================================================================================================
* (b) Ask an LLM, such as ChatGPT, what numerical method is the most efficient approach to compute the stationary distribution? You need to show which prompts are used, and state how you verify the correctness of the results (output by LLMs)
    *
    The most efficient numerical methods to compute the stationary distribution of a Markov chain often depend on the specific characteristics of the transition matrix (e.g., size, sparsity, etc.). Here are some of the most commonly used methods:

### Numerical Methods for Computing Stationary Distributions

1. **Power Iteration Method**:
   - This iterative method starts with an initial guess for the stationary distribution and repeatedly multiplies it by the transition matrix until convergence is reached.
   - **Efficiency**: Generally efficient for large, sparse matrices.
   - **Prompt Example**: "How can I implement the power iteration method to find the stationary distribution of a Markov chain?"

2. **Direct Solution of Linear Equations**:
   - Set up the balance equations pi P = pi as a system of linear equations along with the normalization condition ∑i pi = 1 and solve them using numerical techniques.
   - **Efficiency**: Effective for smaller matrices; methods like Gaussian elimination can be used.
   - **Prompt Example**: "What numerical techniques can I use to solve the linear equations for the stationary distribution of a Markov chain?"

3. **Matrix Exponential Method**:
   - Use the property that the stationary distribution can be derived from the limit of P^n as n -> ∞ . This can be computed efficiently using matrix exponentiation techniques.
   - **Efficiency**: Good for moderate-sized matrices; may require optimization for very large matrices.
   - **Prompt Example**: "How do I compute the stationary distribution using the matrix exponential of the transition matrix?"

4. **Lanczos Algorithm**:
   - This is a powerful method for finding eigenvalues and eigenvectors of large sparse matrices, often used in conjunction with the power method.
   - **Efficiency**: Very efficient for large-scale problems, particularly when the matrix is sparse.
   - **Prompt Example**: "What is the Lanczos algorithm, and how can it help find the stationary distribution of a large Markov chain?"

5. **Gibbs Sampling**:
   - A Markov Chain Monte Carlo method that can approximate the stationary distribution for high-dimensional systems.
   - **Efficiency**: Effective for complex distributions where traditional methods may fail.
   - **Prompt Example**: "How can Gibbs sampling be used to estimate the stationary distribution of a Markov chain?"

### Verification of Results

Professional papers for verification:

To ensure the correctness of the computed stationary distribution, you can follow these verification steps:

1. **Check Normalization**:
   - Verify that the computed stationary distribution sums to 1: ∑i pi_i = 1.

2. **Balance Equations**:
   - Confirm that the distribution satisfies the balance equations pi P = pi.

3. **Empirical Testing**:
   - Run simulations of the Markov chain for a large number of steps and observe the distribution of states. Compare this empirical distribution with the computed stationary distribution.

4. **Convergence Analysis**:
   - Monitor the convergence of the power iteration or any iterative method. You can track the change in the distribution over iterations and ensure it stabilizes.

5. **Sensitivity Analysis**:
   - Alter parameters in the transition matrix and observe the resulting changes in the stationary distribution, ensuring that it behaves consistently under perturbations.

   How I verified:
  Started by searching the methods mentioned online including and typing in " professional papers".  One of them I found is: https://link.springer.com/chapter/10.1007/978-3-031-30823-9_25

### Prompt

- "What is the most efficient way to compute the stationary distribution of a large sparse Markov chain, and how can I verify its accuracy?"

### Conclusion
The choice of method for computing the stationary distribution depends on the characteristics of the Markov chain being analyzed. Verifying the results is crucial to ensure the accuracy and reliability of the computed distribution.

=======================================================================================================
* (c) Implement the method suggested by the LLM. Use the transition matrix generated in question 2.a as an input to compute its stationary distribution.
    * Please implememt the method `compute_stationary_distribution` below

    

    
* (d) The theory of probability matrix is given in the textbook 6.8, Eigenvalues/eigenvector of nonegtive matrices. Read textbook 6.8 and ask two questions that you are curious most about it
    * How can we interpret the eigenvectors of a non-negative matrix in terms of real-world applications, like Markov chains or population models?
    
      Why do non-negative matrices always have a non-negative eigenvalue, and how is the largest eigenvalue related to the structure of the matrix?
   

In [ ]:
def compute_stationary_distribution(transition_matrix: np.ndarray):
    """
        Compute the stationary distribution of a Markov chain.

        Parameters:
            transition_matrix (np.ndarray): A numpy array representing the transition probabilities between states.
        Returns:
            The stationary distribution of the Markov chain.
    """
    # [TODO] Implement the function to compute the stationary distribution of a Markov chain, using the method suggested by the LLM
    # Define the maximum number of iterations and tolerance for convergence
    max_iter = 10000
    tol = 1e-9

    # Get the number of states
    num_states = transition_matrix.shape[0]

    # Initialize the stationary distribution with a uniform distribution
    stationary_distribution = np.ones(num_states) / num_states

    # Power iteration: keep multiplying the distribution by the transition matrix until it converges
    for i in range(max_iter):
        new_distribution = stationary_distribution @ transition_matrix

        # Check for convergence (using the L2 norm to measure the difference)
        if np.linalg.norm(new_distribution - stationary_distribution) < tol:
            return new_distribution

        stationary_distribution = new_distribution

    return stationary_distribution

In [ ]:

# Compute the stationary distribution for the transition matrix obtained from the previous problem
stationary_distribution = compute_stationary_distribution(transition_matrix)

# Convert the stationary distribution to float16 to prevent something like -1.2345678e-16 but it is actually 0
print(stationary_distribution.astype(np.float16))

[0.002811 0.002811 0.002811 0.002811 0.002811 0.002811 0.002811 0.002811
 0.002811 0.002811 0.002811 0.002811 0.002811 0.002811 0.002811 0.002811
 0.002636 0.002636 0.005447 0.002636 0.002636 0.002636 0.002636 0.002636
 0.002636 0.002636 0.002636 0.002636 0.002636 0.002636 0.002636 0.002636
 0.002647 0.002647 0.002647 0.008095 0.002647 0.002647 0.002647 0.002647
 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647
 0.002647 0.002647 0.002647 0.002647 0.01074  0.002647 0.002647 0.002647
 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647
 0.002647 0.002647 0.002647 0.002647 0.002647 0.01339  0.002647 0.002647
 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647
 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.01604  0.002647
 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647
 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.002647 0.01868
 0.002647 0.002647 0.002647 0.002647 0.002647 0.0026